# [Step 6] Quantization & Edge Deployment: 비행 컴퓨터를 위한 AI 경량화

> 이 노트북은 **GRU 배포** 실습이다. Mamba 는 이론 배경으로만 소개하며, 학습·배포는 Step 4의 GRU로 수행한다.
> 목표: FP32 → INT8 양자화, ONNX export, PyTorch↔ONNX 수치 동등성, 물리 후처리 포함.


# 1. 이론적 배경 1: Mamba (State Space Model) 의 수학적 기초

## 1.1 State Space Equation 의 귀환: 제어공학과 딥러닝의 만남

Step 1 (칼만 필터) 에서 우리는 선형 시스템의 상태방정식 (State Equation) 을 배웠습니다:

$$ \dot{h}(t) = Ah(t) + Bx(t) $$
$$ y(t) = Ch(t) $$

여기서:
- $h(t) \in \mathbb{R}^N$: 잠재 상태 (Latent State) - 시스템의 내부 메모리
- $x(t) \in \mathbb{R}$: 입력 신호 (센서 데이터)
- $y(t) \in \mathbb{R}$: 출력 신호 (아포지 예측)
- $A, B, C$: 시스템 행렬 (전통적으로는 수동 설계)

놀라운 사실: Mamba (Gu & Dao, 2023) 는 바로 이 전통적인 제어공학의 상태방정식을 딥러닝의 핵심 구조로 차용합니다.

## 1.2 연속시간에서 이산시간으로: Discretization

컴퓨터는 연속시간을 처리할 수 없으므로, 영점 유지 (Zero-Order Hold, ZOH) 를 사용하여 이산화합니다:

$$ \bar{A} = \exp(\Delta A) $$
$$ \bar{B} = (\Delta A)^{-1} (\exp(\Delta A) - I) \cdot \Delta B $$

이산시간 상태방정식:

$$ h_t = \bar{A}h_{t-1} + \bar{B}x_t $$
$$ y_t = Ch_t $$

여기서 $\Delta$ 는 timescale parameter로, 각 입력 스텝의 시간 간격을 나타냅니다.

> 기계공학적 통찰:
> "우리가 Step 1 에서 칼만 필터의 상태 전이 행렬 $F$ 를 수동으로 설계했던 것과 달리, 
> Mamba 는 데이터로부터 최적의 $\bar{A}, \bar{B}, C$ 를 스스로 학습합니다. 
> 이것이 바로 '데이터 기반 필터'의 핵심입니다."

## 1.3 Mamba 의 핵심 혁신: Selection Mechanism

기존 SSM (S4) 의 한계는 시간 불변 (Time-Invariant) 이었다는 점입니다:
- $\bar{A}, \bar{B}, C$ 가 모든 시간 스텝에서 고정됨
- 입력 내용에 따라 정보를 선택적으로 기억하거나 잊을 수 없음

Mamba 의 해결책 (Gu & Dao, 2023, Section 3.2):

$$ B_t = s_B(x_t), \quad C_t = s_C(x_t), \quad \Delta_t = \mathrm{softplus}(s_\Delta(x_t)) $$
$$ \bar{A}_t = \exp(\Delta_t A), \quad \bar{B}_t = (\Delta_t A)^{-1}(\exp(\Delta_t A)-I)\Delta_t B_t $$

입력 의존 연속 파라미터 $B_t, C_t, \Delta_t$ 와 이산화 결과 $\bar A_t, \bar B_t$ 를 구분한다.

즉, SSM 파라미터를 입력의 함수로 만듭니다. 이로 인해:

| 항목 | 기존 SSM (S4) | Mamba (Selective SSM) |
|------|--------------|----------------------|
| 시간 특성 | Time-Invariant (LTI) | Time-Varying |
| 선택성 | 입력 내용에 따른 선택 없음 | 내용 기반 선택 |
| 계산 모드 | 재귀 또는 합성곱 | Recurrent (Scan) |
| 정보 처리 | 고정 LTI 필터 (시간·주파수별 이득은 가능) | 입력 의존 필터 |

## 1.4 하드웨어 인식 알고리즘: 효율성의 비결

Mamba 는 Selective Scan이라는 하드웨어 인식 알고리즘을 사용합니다 (Gu & Dao, 2023, Section 3.3):

1. Kernel Fusion: 여러 연산을 하나의 GPU 커널로 융합
2. Parallel Scan: 순차적 계산을 병렬화
3. Recomputation: 중간 상태 저장 대신 재계산으로 메모리 절약

결과 (Gu & Dao, 2023 의 효율성 벤치마크, 특정 GPU 조건):
- 학습: 특정 GPU scan 구현에서 표준 SSM 구현 대비 큰 가속을 보고 (보편적 40배 아님)
- 추론: 특정 언어모델·배치 조건의 스루풋 비교 (이 GRU/MCU 실습에 적용되지 않음)
- 메모리: 시퀀스 길이에 대해 선형 스케일링 O(N)

## 1.5 Vision Mamba: 이미지/시계열 데이터 적용

Zhu et al. (2024) 은 Mamba 를 비전 작업에 적용한 Vision Mamba (Vim) 를 제안했습니다:

주요 기여:
1. Bidirectional SSM: 순방향 + 역방향 상태 공간 모델링으로 글로벌 컨텍스트 포착
2. Position Embeddings: 공간 정보 인식을 위한 위치 인코딩 추가
3. Patch Tokenization: 이미지를 패치 시퀀스로 변환하여 Mamba 에 입력

성능 (Zhu et al., 2024, Figure 1):
- DeiT 대비 2.8 배 빠름 (1248×1248 고해상도 이미지)
- GPU 메모리 86.8% 절약
- ImageNet 분류에서 DeiT 보다 높은 정확도

> 로켓 아포지 예측에의 시사점:
> Vision Mamba 의 bidirectional SSM 은 **오프라인 분석**에서 과거-현재-미래를 모두 쓸 수 있다.
> 온라인 예측에서는 예측 시각 이후 관측을 쓰면 미래 누출이다. 다만 현재까지의 관측만 담은
> window 내부를 양방향 처리하는 것은 가능하며, 단방향 상태 갱신과 달리 창 재계산 비용·지연을
> 별도 검증해야 한다. "bidirectional이 항상 더 정확하다"는 실시간 온보드 주장은 아니다.

---

## 참고 문헌 (Cell 2)

1. Gu, A., & Dao, T. (2023). Mamba: Linear-Time Sequence Modeling with Selective State Spaces. *arXiv:2312.00752*.
   - Section 2: State Space Models (수학적 기초)
   - Section 3: Selective State Space Models (Selection Mechanism)
   - Section 3.3: Efficient Implementation (하드웨어 인식 알고리즘)

2. Zhu, L., et al. (2024). Vision Mamba: Efficient Visual Representation Learning with Bidirectional State Space Model. *arXiv:2401.09417*.
   - Section 3: Method (Vim 아키텍처)
   - Section 3.5: Efficiency Analysis (효율성 분석)

# 2. 이론적 배경 2: Mamba vs GRU vs Transformer 비교

## 2.1 시계열 모델의 진화

| 세대 | 모델 | 연산 복잡도 (학습) | 추론 메모리 | 장기 의존성 | 온보드 적합성 |
|------|------|------------|--------|------------|--------------|
| 1 세대 | RNN (tanh) | O(N) | O(1) | 약함 (Vanishing Gradient) | ✅ |
| 2 세대 | LSTM/GRU | O(N) | O(1) | 게이트로 수백 스텝 유지 가능 | ✅ |
| 3 세대 | Transformer | O(N²) (attention) | O(N) (KV cache) | 강함 | ❌ |
| 4 세대 | Mamba | O(N) (scan) | O(1) (고정 상태) | 유한 상태로 근사 | ✅ |

## 2.2 연산 복잡도 분석

### Transformer 의 한계: 자기Attention 의 제곱 스케일링

$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V $$

- $Q, K, V \in \mathbb{R}^{N \times d}$ (N: 시퀀스 길이, d: 임베딩 차원)
- $QK^T$ 연산: O(N²d) - 시퀀스 길이가 2 배가 되면 연산량 4 배 증가
- 로켓 비행 (500 스텝): 250,000 개의 attention score 계산 필요

### Mamba 의 장점: 선형 스케일링

$$ h_t = \bar{A}_t h_{t-1} + \bar{B}_t x_t $$

- 각 스텝당 상수 시간 O(1) 연산
- 전체 시퀀스: O(N) - 시퀀스 길이가 2 배가 되면 연산량 2 배 증가
- 로켓 비행 (500 스텝): 500 회의 상태 업데이트만 필요

> Gu & Dao (2023) 의 언어모델 실험 (보고된 조건):
> - Mamba-3B 는 Pythia-7B (2 배 큰 Transformer) 와 동등한 성능
> - 추론 스루풋: 특정 언어모델·배치 조건에서 Transformer 대비 우위를 보고 (수치는 조건 의존)

## 2.3 장기 의존성 (Long-Term Dependency) 비교

### GRU 의 한계 (Chung et al., 2014)

$$ h_t = z_t \odot h_{t-1} + (1 - z_t) \odot n_t \quad \text{(PyTorch 관례)} $$

- Update Gate ($z_t$) 가 1에 가까우면 이전 상태를 유지한다.
- 원 논문(Chung et al.)의 상보적 표기와 reset gate 적용 위치는 PyTorch와 다르다.
- 장기 의존성은 **경향성**의 문제다. 200 스텝이 hard limit 은 아니며, 게이트가 1에 가까우면 수백~수천 스텝도 유지될 수 있다.

### Mamba 의 장점: Selective Memory

$$ h_t = \exp(\Delta_t A) h_{t-1} + (\Delta_t A)^{-1} (\exp(\Delta_t A) - I) \cdot \Delta_t B_t x_t $$

- $\Delta_t$ (timescale) 가 입력에 따라 동적으로 조절됨
- 기존 기억을 유지하려면 $\Delta_t$ 를 작게, 현재 입력으로 상태를 갱신하려면 크게 학습
- 유한 차원 상태이므로 "무제한 기억"은 아니고, 고정 크기 상태로 장기 의존성을 근사한다

> Gu & Dao (2023) 의 Induction Heads 실험 (보고된 조건):
> - Mamba 는 100 만 토큰 (1M tokens) 시퀀스에서도 100% 정확도 유지
> - Transformer 는 16K 토큰에서 메모리 부족 (OOM)
> - GRU/Hyena 는 2 배 길이 이상에서 성능 급감

---

## 참고 문헌 (Cell 3)

1. Gu, A., & Dao, T. (2023). Mamba: Linear-Time Sequence Modeling with Selective State Spaces. *arXiv:2312.00752*.
   - Table 3: Induction Heads Extrapolation (장기 의존성)
   - Figure 8: Efficiency Benchmarks (속도/메모리)

2. Zhu, L., et al. (2024). Vision Mamba: Efficient Visual Representation Learning with Bidirectional State Space Model. *arXiv:2401.09417*.
   - Figure 1: Performance and Efficiency Comparisons
   - Table 1: ImageNet Classification Results

3. Chung, J., et al. (2014). Empirical Evaluation of Gated Recurrent Neural Networks on Sequence Modeling. *arXiv:1412.3555*.
   - Section 3.2: Gated Recurrent Unit (GRU 구조)

# 3. 이론적 배경 3: 모델 양자화 (Quantization) 원리

## 3.1 왜 양자화가 필요한가?

### 임베디드 시스템의 제약 (Avionics Constraints)

| 항목 | 지상 PC (학습) | 비행 컴퓨터 (추론) |
|------|--------------|------------------|
| 프로세서 | GPU (RTX 4090) | Edge (RPi) / MCU (STM32, 별도 툴체인) |
| 메모리 | 32GB RAM | 256MB RAM |
| 정밀도 | FP32 (32-bit) | INT8 (8-bit) 선호 |
| 전력 | 450W | 5W |
| 프레임워크 | PyTorch | C++ / ONNX Runtime |

문제: FP32 모델을 그대로 탑재하면:
- 메모리 부족으로 로드 불가
- 추론 속도 50Hz 요구사항 미달 (0.1 초 → 0.5 초)
- 전력 소모 배터리 고갈

## 3.2 양자화의 수학적 기초

### FP32 → INT8 변환

양자화 (Quantization):

$$ x_{int8} = \text{round}\left(\frac{x_{fp32}}{s}\right) + z $$

역양자화 (Dequantization):

$$ x_{fp32} \approx s \cdot (x_{int8} - z) $$

여기서:
- $s$: scale factor (실수 → 정수 비율)
- $z$: zero point (0 의 정수 표현)
- $x_{int8} \in [-128, 127]$ (8-bit 부호정수)

### 양자화 오차 (Quantization Error)

$$ \text{Error} = |x_{fp32} - \hat{x}_{fp32}| \leq \frac{s}{2} $$

> 이 bound는 포화가 없는 **단일 값**의 최근접 반올림에만 적용된다. 가중치·중간 활성화 오차가
> 누적되는 신경망 출력 전체의 상한이 아니며, 최종 r을 고정 간격 s로 직접 양자화하는 실험도 아니다.

로켓 아포지 예측에서:
- 여기서 $s$ 는 **무차원 $r$** 의 양자화 간격이다 (예: $s \approx 0.01$ → $r$ 오차 ±0.005).
- 현재 head에서 같은 h·v를 쓰면 $\Delta H = (h+G)\cdot\Delta r_{eff}$ 이다. $\Delta r \times v_z^2/2g$ 는 잘못된 식이다.
- 아래 $s$ 예시는 **r 자체를 고정 간격으로 직접 양자화한다고 가정한 별도 예시**다. 현재 실험의 양자화 차이는 실제 창의 최종 출력을 직접 비교한다.

## 3.3 양자화 방식 비교

| 방식 | 설명 | 장점 | 단점 | 적합성 |
|------|------|------|------|--------|
| Post-Training (PTQ) | 학습 후 양자화 | 재학습 불필요, 빠름 | 정확도 손실 가능 | ✅ 권장 |
| Quantization-Aware (QAT) | 학습 중 양자화 | 정확도 높음 | 재학습 필요, 느림 | ⚠️ |
| Dynamic | 가중치 INT8, 활성화는 실행 시 동적 양자화 | 간단, 호환성 좋음 | 압축률·지연은 측정 필요 | ✅ Step 6 사용 |
| Static | 가중치 + 활성화 INT8 | 이론상 더 큰 절감 | 보정 데이터 필요, 연산 지원 제약 | ⚠️ |

## 3.4 PyTorch 동적 양자화 메커니즘

```python
# PyTorch 동적 양자화 (torch.ao.quantization)
quantized_model = torch.ao.quantization.quantize_dynamic(
    model,                    # 원본 모델
    {nn.GRU, nn.Linear},      # 양자화할 레이어
    dtype=torch.qint8         # 8-bit 부호정수
)
```

동작 원리:
1. 가중치: FP32 → INT8로 미리 저장
2. 활성화: FP32 입출력, 지원 연산 내부에서 실행 시 양자화
3. 연산: 정수 행렬 연산 결과를 FP32로 반환하며 일부 연산은 FP32로 남음

장점:
- 재학습 불필요 (바로 적용 가능)
- 호환성 좋음 (기존 PyTorch 코드 변경 최소화)
- 정확도 손실은 모델·입력에 따라 다르므로 실제 창에서 측정해야 함

## 3.5 양자화 모델의 온보드 배포

### ONNX (Open Neural Network Exchange)

왜 ONNX 인가?
- 플랫폼 독립적: PyTorch, TensorFlow, C++ 모두 지원
- 최적화 가능: ONNX Runtime 에서 자동 최적화
- 모바일 지원: ONNX Runtime mobile (Android/iOS 등). MCU 경로는 별도 변환·툴체인 검증 필요

변환 과정:

```
PyTorch (FP32) 
    ↓ torch.onnx.export  (이 실습에서 수행)
ONNX (FP32) 
    ↓ 별도 INT8 ONNX 변환 (이 실습 범위 밖)
ONNX (INT8, 개념도)
    ↓ ONNX Runtime (host/mobile) 또는 타깃 툴체인
호스트 검증 → MCU 변환·컴파일·탑재는 별도
```

### 예상 성능 개선 (실측 전 — 출처 없음)

아래 표는 **가정**이며 Gu & Dao (2023) 의 수치가 아니다. 실제 값은 이 노트북의
`[Cell 4]`~`[Cell 5]` 에서 측정한다. 온보드 지연은 MCU/NPU 사양에 따라 달라진다.

| 항목 | FP32 | INT8 (가정) | 개선 |
|------|------|------|------|
| 모델 크기 | 측정값 | 측정값 | 파일 크기 기준 |
| 추론 속도 | 측정값 | 측정값 | 이 PC CPU 기준 |
| 정확도 손실 | - | 측정값 | 실제 test 창에서 |

> 엔지니어링적 타협:
> 양자화 오차는 실제 test 창에서 FP32/INT8 최종 head 출력을 직접 비교한다. 별도 가정으로 r을 고정 간격 s로 양자화할 때만 $\Delta H=(h+G)\Delta r_{eff}$ 로 환산한다.
> 메모리 절약과 지연은 온보드 타깃에서 별도로 측정해야 한다.

---

## 참고 문헌 (Cell 4)

1. Gu, A., & Dao, T. (2023). Mamba: Linear-Time Sequence Modeling with Selective State Spaces. *arXiv:2312.00752*.
   - Section 4.5: Speed and Memory Benchmarks (효율성 벤치마크)

2. PyTorch Documentation. Quantization. *https://pytorch.org/docs/stable/quantization.html*
   - torch.ao.quantization.quantize_dynamic (동적 양자화 API)

3. ONNX Documentation. *https://onnx.ai/*
   - ONNX Runtime for Edge Devices (임베디드 배포)
```

### 위에서는 MAMBA 에 대해 개념적으로 설명했지만, 학습 난이도와 시간상의 이유로 
### Step4,5에서 완성해놓은 GRU를 바탕으로 양자화, 배포 할 것임

In [1]:
# ============================================================================
# [Cell 1 & 2] 라이브러리 및 GRU 모델 클래스 정의
# ============================================================================
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import os
import time
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# 재현성
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cpu') # 양자화 및 ONNX는 CPU 기반 작업이 많음
print(f"작업 환경: {device}")

# --- Step 4와 동일한 모델 구조 정의 (필수) ---
class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, output_dim=1, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, 
                          batch_first=True, dropout=dropout if num_layers > 1 else 0)
        
        # Step 4와 인덱스를 맞추기 위해 Dropout을 포함시킴
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 64), # fc.0
            nn.ReLU(),                 # fc.1
            nn.Dropout(dropout),       # fc.2 (여기가 추가되어야 함!)
            nn.Linear(64, output_dim), # fc.3
            nn.Tanh()                  # fc.4 → 1+0.5*tanh
        )
    
    def forward(self, x, h_curr=None, v_z=None):
        gru_out, _ = self.gru(x)
        last_hidden = gru_out[:, -1, :]
        # r = 1 + 0.5*tanh ∈ (0.5, 1.5)
        r_pred = 1.0 + 0.5 * self.fc(last_hidden).squeeze(-1)
        
        if h_curr is not None and v_z is not None:
            h_theo_gain = (v_z ** 2) / (2 * 9.81)
            denom = h_curr + h_theo_gain
            if not torch.isfinite(denom).all() or not (denom > 0).all():
                raise ValueError("h + v^2/(2g) must be finite and positive")
            h_apogee = denom * r_pred
            return h_apogee, r_pred
        return r_pred

print("GRU 모델 클래스 정의 완료")

작업 환경: cpu
GRU 모델 클래스 정의 완료


In [2]:
# ============================================================================
# [Cell 3] Step 4에서 학습한 Best Model 로드 (Key 매핑 수정)
# ============================================================================
from collections import OrderedDict

input_dim = 6 
model_fp32 = GRUModel(input_dim=input_dim).to(device)

model_path = './data/simulated/best_gru_model.pth'

if os.path.exists(model_path):
    # 1. 저장된 state_dict 불러오기
    state_dict = torch.load(model_path, map_location=device)
    
    # 2. 키 이름 변경 (Wrapper의 'gru.' 접두사 제거)
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        # 'gru.' 로 시작하면 그 뒤의 이름만 사용
        name = k[4:] if k.startswith('gru.') else k
        new_state_dict[name] = v
        
    # 3. 변경된 state_dict로 모델 로드
    try:
        model_fp32.load_state_dict(new_state_dict)
        print("학습된 GRU 모델 가중치 로드 성공! (Prefix 제거됨)")
    except RuntimeError as e:
        print(f"가중치 로드 실패: {e}")
        print("키 이름이 맞는지 확인해주세요.")
else:
    raise FileNotFoundError("Step 4를 먼저 실행하여 모델을 학습시켜야 함.")

model_fp32.eval() # 평가 모드 전환

# 파라미터 수 확인
fp32_params = sum(p.numel() for p in model_fp32.parameters())
fp32_size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"   - 파라미터 수: {fp32_params:,} 개")
print(f"   - 파일 크기: {fp32_size_mb:.2f} MB")

학습된 GRU 모델 가중치 로드 성공! (Prefix 제거됨)
   - 파라미터 수: 159,617 개
   - 파일 크기: 0.61 MB


In [3]:
# ============================================================================
# [Cell 4] Dynamic Quantization (FP32 -> INT8)
# ============================================================================
import torch.ao.quantization

print("모델 경량화(Quantization) 시작...")

# 1. 양자화 적용 (Linear, GRU 레이어를 INT8로 변환)
# PyTorch는 동적 양자화를 통해 가중치만 INT8로 줄이고, 
# 연산 시점에 활성화 함수를 양자화하여 속도와 크기를 모두 잡음.
model_int8 = torch.ao.quantization.quantize_dynamic(
    model_fp32,  # 원본 FP32 모델
    {nn.Linear, nn.GRU},  # 양자화 대상 레이어
    dtype=torch.qint8
)

print("양자화 완료")

# 2. 크기 비교를 위해 저장
torch.save(model_int8.state_dict(), './data/simulated/gru_int8.pth')
int8_size_mb = os.path.getsize('./data/simulated/gru_int8.pth') / (1024 * 1024)

# 3. 결과 리포트
print(f"\n[경량화 성과 보고]")
print(f"   - Original (FP32): {fp32_size_mb:.2f} MB")
print(f"   - Quantized (INT8): {int8_size_mb:.2f} MB")
print(f"   - 압축률: {fp32_size_mb / int8_size_mb:.1f}배 (메모리 {100*(1 - int8_size_mb/fp32_size_mb):.1f}% 절약)")

모델 경량화(Quantization) 시작...
양자화 완료

[경량화 성과 보고]
   - Original (FP32): 0.61 MB
   - Quantized (INT8): 0.16 MB
   - 압축률: 3.8배 (메모리 73.4% 절약)


C:\Users\Seungwon Lee\projects\PSIntelligence\psintelligence\.venv\lib\site-packages\torch\ao\nn\quantized\dynamic\modules\rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\quantized\Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


In [4]:
# ============================================================================
# [Cell 5] 성능 벤치마크 (FP32 vs INT8)
# ============================================================================
# 더미 데이터 생성 (Batch=1, Seq=100, Feat=6)
dummy_input = torch.randn(1, 100, 6).to(device)

def measure_inference(model, input_data, n_loops=1000):
    start = time.time()
    for _ in range(n_loops):
        with torch.no_grad():
            _ = model(input_data)
    end = time.time()
    return (end - start) / n_loops * 1000 # ms 단위

# 1. 속도 측정
time_fp32 = measure_inference(model_fp32, dummy_input)
time_int8 = measure_inference(model_int8, dummy_input)

print(f"\n[추론 속도 비교 (CPU 기준)]")
print(f"  - FP32 Latency: {time_fp32:.3f} ms")
print(f"  - INT8 Latency: {time_int8:.3f} ms")
print(f"  - 속도 향상: {time_fp32 / time_int8:.1f}배")

# 2. 출력 오차 검증 (실제 test 비행 창에서)
import joblib

import apogee_pipeline as ap
FEATURES = ['altitude_noisy_rel', 'velocity_z', 'acceleration_inertial',
            'tilt_angle', 'gyro_roll_noisy', 'dynamic_pressure']
scaler = joblib.load('./data/simulated/input_scaler.pkl')
scaler = ap.adapt_legacy_scaler_to_inertial(scaler)
df_test = pd.read_csv('./data/simulated/all_trajectories.csv.gz')
_pad = df_test.groupby('flight_id')['altitude'].transform('first')
df_test['altitude_rel'] = df_test['altitude'] - _pad
df_test['altitude_noisy_rel'] = df_test['altitude_noisy'] - _pad
df_test['acceleration_inertial'] = df_test['acceleration_noisy'] - ap.SIM_G
if 'dynamic_pressure' not in df_test.columns:
    _rho = 1.225 * np.exp(-0.00012 * df_test['altitude_rel'].abs())
    df_test['dynamic_pressure'] = 0.5 * _rho * (df_test['velocity_z'] ** 2)

# common split manifest (calibration/test are disjoint 25-flight sets)
flight_ids = df_test['flight_id'].unique()
split = ap.build_split_manifest(flight_ids, seed=42)
test_ids = split['test']

seqs, hs, vs = [], [], []
for fid in test_ids[:5]:
    g = df_test[df_test['flight_id'] == fid].reset_index(drop=True)
    for i in range(0, len(g) - 100 + 1, 10):
        seqs.append(scaler.transform(g.iloc[i:i+100][FEATURES].values))
        hs.append(g.iloc[i+99]['altitude_rel'])
        vs.append(g.iloc[i+99]['velocity_z'])
x_real = torch.FloatTensor(np.array(seqs))
h_real = torch.FloatTensor(np.array(hs))
v_real = torch.FloatTensor(np.array(vs))

with torch.no_grad():
    h_fp32, r_fp32 = model_fp32(x_real, h_real, v_real)
    h_int8, r_int8 = model_int8(x_real, h_real, v_real)

diff_r = torch.abs(r_fp32 - r_int8).max().item()
diff_h = torch.abs(h_fp32 - h_int8).max().item()
print(f"\n[정확도 검증 — 실제 test 창 {len(x_real)}개]")
print(f"   - r 최대 차이: {diff_r:.6f}")
print(f"   - 아포지 최대 차이: {diff_h:.4f} m  ((h+G)·Δr)")
print(f"   - 참고: 더미 입력 차이 {torch.abs(model_fp32(dummy_input) - model_int8(dummy_input)).max().item():.6f}")


[추론 속도 비교 (CPU 기준)]
  - FP32 Latency: 4.026 ms
  - INT8 Latency: 6.541 ms
  - 속도 향상: 0.6배



[정확도 검증 — 실제 test 창 117개]
   - r 최대 차이: 0.010696
   - 아포지 최대 차이: 1.6579 m  ((h+G)·Δr)
   - 참고: 더미 입력 차이 0.000157


## 5.1 [분석] 모델 경량화(Quantization) 결과의 공학적 해석

우리는 방금 FP32(32비트 실수) 모델을 INT8(8비트 정수) 모델로 변환하는 '동적 양자화'를 수행하였음. 이 결과가 실제 로켓 에비오닉스 설계에 미치는 영향을 세 가지 관점에서 분석해볼 수 있다.

### 1. 메모리 점유율 (Storage & RAM Efficiency)
*   성과: 모델 파일 크기가 0.61 MB → 0.16 MB (73.4% 감소, 3.8배 압축).
*   의미: 
    *   일반적인 PC 환경에서는 수 MB 차이가 작지만, STM32나 Teensy와 같은 MCU에서는 큰 차이임.
    *   하드웨어의 내장 플래시와 SRAM은 제한적이므로, 이 압축이 온보드 적재 가능성을 결정한다.

### 2. 추론 속도와 하드웨어의 특성 (Latency Trade-off)
*   성과: 이 PC CPU에서 FP32 약 4.1 ms → INT8 약 6.6 ms (0.6배, 오히려 느림). 실행마다 수십 % 변동한다.
*   원인 분석: 
    *   PC CPU는 FP32 연산에 최적화되어 있고, 동적 양자화는 연산 시점 변환 오버헤드가 있다.
    *   모델이 작아 정수 연산 이득보다 변환 비용이 클 수 있다.
*   온보드에서는 NPU/정수 연산 유닛 유무에 따라 결과가 달라지므로, **타깃 MCU에서 별도 측정**해야 한다.
  이 PC의 약 6.6 ms가 20 ms 이내라고 해서 STM32에서도 50Hz가 보장되지는 않는다.

### 3. 수치적 정밀도 (Numerical Accuracy)
*   성과: 실제 test 창 117개에서 FP32↔INT8 r 최대 차이 0.010696, 아포지 최대 차이 1.66 m.
*   의미: 
    *   r 차이는 `ΔH = (h+G)·Δr_eff` 로 환산한다. 여기서 r_eff는 clamp 후 반환값이다 (위 1.66 m는 head 출력 차이를 직접 계산한 값).
    *   센서 노이즈(기압계 ±1.5m)와 비교해 허용 가능한지 판단하라.

---

### 요약 
> "FP32 대비 아포지 추가 차이 1.18 m를 유지하면서도, 메모리 점유율은 1/4 수준으로 낮추고, 이 PC CPU에서 20 ms 이내로 추론"

In [5]:
# ============================================================================
# [Cell 6] ONNX Export — 아포지와 r 을 함께 내보낸다
# ============================================================================
import torch.onnx
import onnx

onnx_path = "./data/simulated/rocket_gru.onnx"
print(f"ONNX 변환 시작: {onnx_path}")

dummy_x = torch.randn(1, 100, 6)
dummy_h = torch.tensor([100.0])
dummy_v = torch.tensor([50.0])

torch.onnx.export(
    model_fp32,
    (dummy_x, dummy_h, dummy_v),
    onnx_path,
    export_params=True,
    opset_version=18,
    do_constant_folding=True,
    dynamo=False,  # legacy exporter (onnxscript 불필요)
    input_names=['input_sequence', 'h_curr', 'v_z'],
    output_names=['apogee', 'energy_ratio'],
    dynamic_axes={
        'input_sequence': {0: 'batch_size', 1: 'seq_len'},
        'h_curr': {0: 'batch_size'},
        'v_z': {0: 'batch_size'},
        'apogee': {0: 'batch_size'},
        'energy_ratio': {0: 'batch_size'},
    }
)

# 생성된 파일 검증 (실패하면 다음 셀로 진행하지 않는다)
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print(f"\n[ONNX Export 완료]")
print(f"   - 저장 경로: {onnx_path}")
print(f"   - Opset: 18")
print(f"   - 출력: apogee, energy_ratio (물리 후처리 포함)")
print(f"   - 모델 검증: 통과")

ONNX 변환 시작: ./data/simulated/rocket_gru.onnx

[ONNX Export 완료]
   - 저장 경로: ./data/simulated/rocket_gru.onnx
   - Opset: 18
   - 출력: apogee, energy_ratio (물리 후처리 포함)
   - 모델 검증: 통과


C:\Users\Seungwon Lee\projects\PSIntelligence\psintelligence\.venv\lib\site-packages\torch\onnx\_internal\torchscript_exporter\symbolic_opset9.py:4604: UserWarning: Exporting a model to ONNX with a batch_size other than 1, with a variable length with GRU can cause an error when running the ONNX model with a different batch size. Make sure to save the model with a batch size of 1, or define the initial states (h0/c0) as inputs of the model. 
  return _generic_rnn(


In [6]:
import onnxruntime as ort

# 추론 세션 생성
session = ort.InferenceSession(onnx_path)
input_names = [i.name for i in session.get_inputs()]
print(f"ONNX 입력: {input_names}")

# 같은 입력으로 PyTorch 와 ONNX 출력을 비교한다 (수치 동등성)
test_x = torch.randn(2, 100, 6)
test_h = torch.tensor([100.0, 80.0])
test_v = torch.tensor([50.0, 30.0])

feed = {
    'input_sequence': test_x.numpy().astype(np.float32),
    'h_curr': test_h.numpy().astype(np.float32),
    'v_z': test_v.numpy().astype(np.float32),
}
onnx_apogee, onnx_r = session.run(None, feed)

with torch.no_grad():
    torch_apogee, torch_r = model_fp32(test_x, test_h, test_v)

err_h = float(np.max(np.abs(onnx_apogee.ravel() - torch_apogee.numpy().ravel())))
err_r = float(np.max(np.abs(onnx_r.ravel() - torch_r.numpy().ravel())))

print(f"\n[PyTorch ↔ ONNX 수치 동등성]")
print(f"   - 아포지 최대 차이: {err_h:.6f} m")
print(f"   - r 최대 차이:      {err_r:.8f}")
print(f"   - ONNX 아포지 샘플: {onnx_apogee.ravel()[0]:.2f} m")
print(f"   - ONNX r 샘플:      {onnx_r.ravel()[0]:.4f}")
print("   → 이 .onnx 하나로 물리 후처리까지 끝난다. C++에서 r 을 다시 계산할 필요가 없다.")

ONNX 입력: ['input_sequence', 'h_curr', 'v_z']

[PyTorch ↔ ONNX 수치 동등성]
   - 아포지 최대 차이: 0.000031 m
   - r 최대 차이:      0.00000012
   - ONNX 아포지 샘플: 180.27 m
   - ONNX r 샘플:      0.7927
   → 이 .onnx 하나로 물리 후처리까지 끝난다. C++에서 r 을 다시 계산할 필요가 없다.
